# 01 — Auto-Labeling with Gemini 1.5 Flash

Uses Gemini Flash API to automatically extract structured labels
from financial document pages. These labels become training data
for fine-tuning Qwen3-VL.

**Cost**: ~₹500 for 1,150 PDFs (34,500 pages)


In [ ]:
# Cell 1: Configure Gemini API
import google.generativeai as genai
import os, json, time
from pathlib import Path
from PIL import Image

# Set your API key (from Google AI Studio)
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', 'your-key-here')
genai.configure(api_key=GEMINI_API_KEY)

model = genai.GenerativeModel('gemini-1.5-flash')
print('Gemini configured ✅')


In [ ]:
# Cell 2: Labeling prompt
LABELING_PROMPT = '''Analyze this financial statement page and extract ALL tables.

For EACH table, provide:
{
  "table_id": "page{page}_table{n}",
  "zone_type": "BALANCE_SHEET|INCOME_STATEMENT|CASH_FLOW|NOTES_FIXED_ASSETS|NOTES_RECEIVABLES|NOTES_DEBT|NOTES_OTHER|OTHER",
  "zone_label": "Human readable name",
  "confidence": 0.95,
  "periods": ["2025", "2024"],
  "currency": "USD",
  "unit": "millions",
  "headers": ["Account", "2025", "2024"],
  "rows": [
    {"label": "Revenue", "values": [12500, 11200], "is_total": false, "indent_level": 0}
  ]
}

Return JSON: {"page_number": N, "tables": [...], "page_type": "financial_statement|notes|cover|text_only"}'''


In [ ]:
# Cell 3: Process all page images
import glob

IMAGES_DIR = Path('/content/drive/MyDrive/numera-ml/data/page_images')
LABELS_DIR = Path('/content/drive/MyDrive/numera-ml/data/labels')

image_files = sorted(glob.glob(str(IMAGES_DIR / '**/*.png'), recursive=True))
print(f'Found {len(image_files)} page images to label')

errors = []
for i, img_path in enumerate(image_files):
    label_path = LABELS_DIR / Path(img_path).relative_to(IMAGES_DIR).with_suffix('.json')
    if label_path.exists():
        continue  # Already labeled

    try:
        img = Image.open(img_path)
        response = model.generate_content(
            [LABELING_PROMPT, img],
            generation_config={'response_mime_type': 'application/json'}
        )
        label_data = json.loads(response.text)

        label_path.parent.mkdir(parents=True, exist_ok=True)
        label_path.write_text(json.dumps(label_data, indent=2))

        if (i + 1) % 100 == 0:
            print(f'Labeled {i+1}/{len(image_files)}')

        time.sleep(0.5)  # Rate limit
    except Exception as e:
        errors.append((img_path, str(e)))
        time.sleep(2)

print(f'Done! Errors: {len(errors)}')


In [ ]:
# Cell 4: Quality check — sample 20 labels
import random

label_files = sorted(glob.glob(str(LABELS_DIR / '**/*.json'), recursive=True))
samples = random.sample(label_files, min(20, len(label_files)))

for f in samples[:5]:
    data = json.loads(Path(f).read_text())
    tables = data.get('tables', [])
    print(f'{Path(f).name}: {len(tables)} tables')
    for t in tables:
        print(f'  - {t.get("zone_type")}: {len(t.get("rows", []))} rows')
